# FoodScholar — Building the Graph

A runnable tour of the `foodscholar` library. **Every section is self-contained** — each one starts with a `bootstrap()` call that builds whatever state it needs, so you can run any section on its own without executing the ones above it.

- **§1 Quickstart** — the 5-line happy path.
- **§2 Ontology** · **§3 Corpus** · **§4 Annotate** — implemented phases, run for real.
- **§5 Graph** · **§6 Inspect** — graph construction (Layer A/B/C builders are still stubs; the section builds the equivalent by hand via `fs.graph`).

Run from the repo root: `conda activate foodscholar && jupyter notebook notebooks/build_graph.ipynb`

## Setup

Run this once. It makes the package importable and defines `bootstrap()` — the helper every section below uses to get a ready `FoodScholar` instance.

**For the agentic NER / LLM tiers** (§4): install `pip install -e '.[annotate,llm]'` and set `GROQ_API_KEY` in the shell *before* launching Jupyter. Without them, sections degrade to the deterministic keyword/lexical path — they still run.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import foodscholar
from foodscholar.logging import configure_logging
configure_logging(level="INFO")
print("foodscholar", foodscholar.__version__)

foodscholar 0.1.0


In [2]:
from foodscholar import FoodScholar, FoodOnAPI
from foodscholar.io.chunk import Chunk
from foodscholar.ontology import load_ontology

# The notebook prefers the REAL FoodOn ontology. Drop the release at
# data/foodon.owl; first load parses with pronto (~minutes) then caches
# to data/foodon_cache.parquet. If it is absent, bootstrap() falls back
# to the tiny test fixture so the notebook still opens — but a real
# showcase wants the real ontology.
REAL_FOODON = REPO_ROOT / "data" / "foodon.owl"
MINI_FOODON = REPO_ROOT / "tests" / "fixtures" / "mini_foodon.obo"
USING_REAL_FOODON = REAL_FOODON.exists()

# A small real-text corpus — stands in for chunks.parquet.
SAMPLE_CHUNKS = [
    Chunk(chunk_id="c1", text="Mediterranean diet rich in olive oil reduces cardiovascular risk.",
          source_doc_id="d1", source_type="abstract", section_type="abstract", year=2024),
    Chunk(chunk_id="c2", text="Whole grain consumption is associated with lower mortality.",
          source_doc_id="d2", source_type="abstract", section_type="results", year=2023),
    Chunk(chunk_id="c3", text="Peanut allergy management guidelines for paediatric populations.",
          source_doc_id="d3", source_type="guide", section_type="guideline", year=2022),
    Chunk(chunk_id="c4", text="Dietary intake of quinoa improves glycemic control in adults with type 2 diabetes.",
        source_doc_id="d4", source_type="abstract", section_type="results", year=2020),
    Chunk(chunk_id="c5", text="Extra virgin olive oil contains polyphenols that reduce inflammation markers.",
        source_doc_id="d5", source_type="abstract", section_type="discussion", year=2019),
    Chunk(chunk_id="c6", text="School-based peanut allergy education reduces accidental exposures among children.",
        source_doc_id="d6", source_type="guide", section_type="guideline", year=2021),
]

# Lazy + memoized: FoodOn is loaded at most ONCE per kernel, and only when a
# section actually asks for it (bootstrap(with_ontology=True), the default).
# The first real-FoodOn load is slow; every later bootstrap() reuses this.
_ONTOLOGY_CACHE: FoodOnAPI | None = None


def _food_ontology() -> FoodOnAPI:
    """Load real FoodOn if present, else the test fixture — memoized.

    Real FoodOn embeds NCBITaxon/CHEBI/PATO terms inline, so it is filtered
    to FOODON: ids; the fixture uses synthetic TEST: ids, so no filter."""
    global _ONTOLOGY_CACHE
    if _ONTOLOGY_CACHE is None:
        if USING_REAL_FOODON:
            terms = load_ontology(REAL_FOODON,
                                  cache_path=REPO_ROOT / "data" / "foodon_cache.parquet")
            _ONTOLOGY_CACHE = FoodOnAPI(terms, prefix_filter=["FOODON:"])
        else:
            _ONTOLOGY_CACHE = FoodOnAPI(load_ontology(MINI_FOODON), prefix_filter=None)
    return _ONTOLOGY_CACHE


def bootstrap(*, with_chunks: bool = False, with_ontology: bool = True) -> FoodScholar:
    """Build a ready in-memory FoodScholar. Every section calls this so it
    can run standalone.

    with_ontology attaches FoodOn (real if available) — pass False in
    sections that do not touch fs.ontology / fs.linker / fs.annotate so the
    ontology is never loaded for them. When attached, the load is memoized:
    the OWL is parsed at most once per kernel, not once per bootstrap().
    with_chunks loads the sample corpus."""
    fs = FoodScholar.in_memory()
    if with_ontology:
        fs.attach_ontology(_food_ontology())
    if with_chunks:
        fs.upsert_chunks(SAMPLE_CHUNKS)
    return fs


print("FoodOn source:", "REAL (data/foodon.owl)" if USING_REAL_FOODON
      else "mini test fixture — drop the real release at data/foodon.owl")
print("bootstrap() ready — ontology is lazy + memoized (loaded once, on first need)")

FoodOn source: REAL (data/foodon.owl)
bootstrap() ready — ontology is lazy + memoized (loaded once, on first need)


## 1. Quickstart

The whole library in a few lines. `FoodScholar.in_memory()` needs no config, no services.

In [ ]:
fs = FoodScholar.in_memory()
fs.upsert_chunks([
    Chunk(chunk_id="c1", text="Mediterranean diet reduces cardiovascular risk.",
          source_doc_id="d1", source_type="abstract", section_type="abstract"),
])
fs.info()

## 2. Ontology

`fs.ontology` is the FoodOn lookup API — labels, synonyms, ancestors, descendants. It backs the linker and (later) the Layer A backbone. Here we use the tiny test fixture; in production `cfg.ontology.foodon_path` points at the real FoodOn `.owl`.

*Self-contained: just run the cell.*

In [ ]:
fs = bootstrap()                       # real FoodOn if data/foodon.owl exists
ont = fs.ontology
print(f"{len(ont)} terms")

# Resolve by name — works on real FoodOn and the fixture alike.
olive_oil = ont.name_to_id("olive oil")
print("olive oil ->", olive_oil)
if olive_oil:
    print("ancestors ->", [ont.id_to_label(a) for a in ont.id_to_ancestors(olive_oil)])
fruit = ont.name_to_id("fruit")
if fruit:
    kids = ont.id_to_descendants(fruit)[:8]
    print(f"fruit has {len(ont.id_to_descendants(fruit))} descendants, e.g.",
          [ont.id_to_label(d) for d in kids])

## 3. Corpus

Chunks are the input to everything. In production: `fs.load_chunks('data/chunks.parquet')`. Here we upsert the synthetic `SAMPLE_CHUNKS` defined in Setup.

*Self-contained: just run the cell.*

In [ ]:
fs = bootstrap(with_chunks=True, with_ontology=False)
for c in fs.chunk_store.scan():
    print(f"  {c.chunk_id} [{c.source_type}/{c.section_type}] {c.text[:55]}...")

## 4. Annotate

The annotate phase has two pluggable pieces — **NER** (find mentions) and the **linker** (resolve each mention to a FoodOn id). They are independently inspectable; the sub-sections below showcase each on its own, then `fs.annotate()` runs the whole phase.

*Self-contained: run the cells in this section in order.*

### 4a. NER — finding mentions

`fs.ner` extracts `Mention` spans from text. Two strategies (`cfg.annotate.ner`):

- **`keyword`** (default here) — deterministic match against FoodOn labels/synonyms. No LLM.
- **`agentic`** — an LLM extracts mentions and classifies each (`entity_type`).

The keyword path runs everywhere. The agentic cell below runs *for real* if the `[llm]` extra + `GROQ_API_KEY` are present, and prints how to enable it otherwise.

In [4]:
fs = bootstrap()
text = "The Mediterranean diet is rich in olive oil; whole grains lower mortality."

# Default: keyword NER — deterministic, offline.
for m in fs.ner.extract(text):
    print(f"  [{m.entity_type:16s}] {m.text!r}  @{m.start}:{m.end}  ({m.ner_model_version})")

  [food            ] 'olive oil'  @34:43  (keyword-ner-v0)
  [food            ] 'whole'  @45:50  (keyword-ner-v0)


In [3]:
# Agentic NER over the loaded corpus. Runs for real when the [llm] extra
# is installed AND GROQ_API_KEY is set in the SHELL before launching Jupyter
# (never hardcode the key in a cell — it would be saved into this file).
import importlib.util as _u
import os
os.environ["GROQ_API_KEY"] = "***REMOVED-GROQ-KEY***"


if _u.find_spec("groq") and os.environ.get("GROQ_API_KEY"):
    from foodscholar.config import LLMConfig, ProviderConfig
    from foodscholar.llm import build_llm
    from foodscholar.annotate import AgenticNER

    llm = build_llm(LLMConfig(
        primary=ProviderConfig(provider="groq", model="llama-3.3-70b-versatile")))
    agentic = AgenticNER(llm)

    # Run it over every chunk loaded in the facade.
    fs_a = bootstrap(with_chunks=True)
    for chunk in fs_a.chunk_store.scan():
        mentions = agentic.extract(chunk.text)
        print(f"{chunk.chunk_id}: {chunk.text}")
        for m in mentions:
            print(f"    [{m.entity_type:16s}] {m.text!r}  @{m.start}:{m.end}")
        if not mentions:
            print("    (no mentions)")
else:
    print("Agentic NER skipped — needs:  pip install -e '.[llm]'")
    print("and GROQ_API_KEY set in the shell before launching Jupyter.")

ontology.cache_hit path=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon_cache.parquet
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:51.461720Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=3 n_returned=3


c1: Mediterranean diet rich in olive oil reduces cardiovascular risk.
    [dietary_pattern ] 'Mediterranean diet'  @0:18
    [food            ] 'olive oil'  @27:36
    [biomarker       ] 'cardiovascular risk'  @45:64


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:51.988214Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


c2: Whole grain consumption is associated with lower mortality.
    [processing      ] 'Whole grain'  @0:11
    [biomarker       ] 'mortality'  @49:58


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:52.688701Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


...

c4: Dietary intake of quinoa improves glycemic control in adults with type 2 diabetes.
    [food            ] 'quinoa'  @18:24
    [biomarker       ] 'glycemic control'  @34:50
    [population      ] 'adults'  @54:60
    [health          ] 'type 2 diabetes'  @66:81


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:53.584325Z [warning  ] agent_ner.mention_not_in_text  mention='extra virgin'
2026-05-19T12:46:53.598713Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=3 n_returned=4


c5: Extra virgin olive oil contains polyphenols that reduce inflammation markers.
    [food            ] 'Extra virgin olive oil'  @0:22
    [nutrient        ] 'polyphenols'  @32:43
    [biomarker       ] 'inflammation markers'  @56:76


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:53.927564Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


c6: School-based peanut allergy education reduces accidental exposures among children.
    [allergen        ] 'peanut allergy'  @13:27
    [population      ] 'children'  @73:81


### 4b. Linker — resolving mentions to FoodOn ids

`fs.linker` maps a mention to an ontology id. It is a tiered cascade — `link.method` shows which tier answered:

| tier | method | needs |
|---|---|---|
| 1 | `lexical_exact` | nothing |
| 2 | `lexical_fuzzy` | nothing |
| 3 | `dense` | SapBERT (`[annotate]` extra) |
| 4 | `llm` | a provider (`[llm]` extra + key) |

Tiers 1-2 always run. The cell below shows them, then builds a dense-tier linker for real if SapBERT is available.

In [ ]:
fs = bootstrap()

# Tiers 1-2 — lexical, always available.
for q in ["olive oil", "olives", "oliv oil", "quinoa"]:
    link = fs.linker.dry_run(q)
    shown = f"{link.ontology_id} [{link.method}, {link.confidence:.2f}]" if link else "None"
    print(f"  {q:12s} -> {shown}")

In [ ]:
# Tier 3 (dense, SapBERT) — runs for real when transformers is installed.
import importlib.util as _u

if _u.find_spec("transformers"):
    from foodscholar.annotate import ThreeTierLinker, SapBERTEmbedder
    from foodscholar.io.ontology import OntologyTerm

    # 'ascorbate' shares no tokens with 'vitamin C' — only the dense tier links them.
    tiny = FoodOnAPI([
        OntologyTerm(id="FOODON:V1", label="vitamin C", synonyms=("ascorbic acid",)),
        OntologyTerm(id="FOODON:O1", label="olive oil"),
    ], prefix_filter=None)
    dense_linker = ThreeTierLinker(tiny, fuzzy_threshold=0.99,
                                   dense_embedder=SapBERTEmbedder(), dense_threshold=0.70)
    link = dense_linker.dry_run("ascorbate")
    print(f"  ascorbate -> {link.ontology_id} [{link.method}, {link.confidence:.2f}]" if link
          else "  ascorbate -> None")
else:
    print("Dense tier skipped — needs:  pip install -e '.[annotate]'  (SapBERT)")

### 4c. The full phase

`fs.annotate()` runs NER → linker → embeddings over every chunk and writes the enriched copies back. `cfg.annotate` (and `config.example.yaml`) decide which NER and which linker tiers are active; `in_memory()` here uses keyword NER + lexical tiers.

In [ ]:
fs = bootstrap(with_chunks=True)
meta = fs.annotate()
print("artifact:", meta.artifact_id, "records:", meta.record_count)
for c in fs.chunk_store.scan():
    methods = sorted({l.method for l in c.entity_links})
    print(f"  {c.chunk_id}: mentions={[m.text for m in c.mentions]}  "
          f"foodon_ids={c.foodon_ids}  tiers={methods}")

## 5. Graph

The three-layer graph: **shelves** (Layer A backbone), **themes** (Layer B), **cards** (Layer C). The Layer A/B/C *builders* are not implemented yet — this section builds an equivalent small graph by hand via `fs.graph`, which is exactly the API those builders will call.

*Self-contained: just run the cells.*

In [ ]:
fs = bootstrap(with_chunks=True)
fs.annotate()

# Layer A (stub) — hand-build 3 shelves. The foods root is a real ontology
# ancestor of olive oil; fall back gracefully if the term isn't found.
olive_oil = fs.ontology.name_to_id("olive oil")
ancestors = fs.ontology.id_to_ancestors(olive_oil) if olive_oil else []
foods_root = ancestors[-1] if ancestors else None   # broadest ancestor
foods_label = fs.ontology.id_to_label(foods_root) if foods_root else "Foods"
fs.graph.add_shelf(shelf_id="s-foods", label=foods_label,
                   facet="foods", depth=0, foodon_id=foods_root)
fs.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                   facet="dietary_patterns", depth=1, parent_shelf_id="s-foods")
fs.graph.add_shelf(shelf_id="s-allergy", label="Food allergies", facet="allergies", depth=1)
fs.graph.attach_chunks(["c1", "c2"], shelf="s-med")
fs.graph.attach_chunks(["c3"], shelf="s-allergy")

# Layer B (stub) — one theme.
fs.graph.add_theme(theme_id="t-olive", label="Olive oil cardiovascular benefits",
                   shelf_ids=["s-med"], discovered_by="leiden", discovery_version="nb")
fs.graph.attach_chunks(["c1"], theme="t-olive")

# Layer C (stub) — one card.
from foodscholar.io.graph import Card
fs.graph.add_card(Card(
    card_id="card-med", target_id="s-med", target_type="shelf",
    title="Mediterranean diet",
    summary="A dietary pattern with strong cardiovascular evidence.",
    evidence_quality="high", cited_chunk_ids=["c1", "c2"],
    llm_model=fs.llm.model_id, prompt_version="v1"))

print("foods root:", foods_label, f"({foods_root})")
print("graph:", fs.graph.summary())

## 6. Inspect the graph

Everything is reachable through `fs.graph` and its handles — shelves, themes, cards, chunks. Re-run §5 first if you jumped straight here.

This cell rebuilds the graph (via §5's steps) so it is also self-contained.

In [ ]:
# Rebuild a small graph so this section stands alone.
fs = bootstrap(with_chunks=True)
fs.annotate()
fs.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                   facet="dietary_patterns", depth=0)
fs.graph.attach_chunks(["c1", "c2"], shelf="s-med")
fs.graph.add_theme(theme_id="t-olive", label="Olive oil benefits",
                   shelf_ids=["s-med"], discovered_by="leiden", discovery_version="nb")

shelf = fs.graph.shelf("s-med")
print("shelf :", shelf.label, "| facet:", shelf.facet)
print("chunks:", [c.chunk_id for c in shelf.chunks()])
print("themes:", [t.label for t in shelf.themes()])
print("search:", [c.chunk_id for c in fs.graph.search("olive oil", shelf="s-med", k=3)])

from foodscholar import make_artifact_meta
meta = make_artifact_meta(phase="notebook-build", config=fs.config, record_count=3)
print("\nartifact:", meta.artifact_id, "| config_hash:", meta.config_hash)

## Status & next

Implemented: ontology loader, the annotate phase (NER + tiered linker + embedders), the `fs.graph` read/write surface, the provider-agnostic LLM layer.

Stubs: the Layer A / B / C *builders* and the retrieval pipeline — §5 builds their output by hand for now. See `docs/DESIGN_agentic_annotate.md` for the agentic-annotation roadmap and `BRIEF.md` §12 for milestone order.